# CrowdWise Delhi — Phase 1 Crowding Pipeline & ML Imputation (Colab)

Runs end-to-end ETL + Two-Tier Trip Reconstruction + LightGBM Sparse-Cell Quantile Imputation + RAPTOR Index Export.

**Inputs:**
- `October-Delhi.csv.zip` (auto-downloaded if not in Drive)
- Static GTFS (`gtfs/extracted/*.txt` in Drive `MyDrive/CrowdWise-Delhi/gtfs/`)

**Outputs (saved to `MyDrive/CrowdWise-Delhi/processed/`):**
- `crowdwise.sqlite`
- `raptor_index.pkl`
- `validation_report.json`

In [ ]:
!pip -q install duckdb rapidfuzz lightgbm scikit-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, math, time, pickle, sqlite3, zipfile
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import duckdb
from rapidfuzz import process, fuzz
import lightgbm as lgb

BASE_DRIVE = Path('/content/drive/MyDrive/CrowdWise-Delhi')
RAW_DIR = BASE_DRIVE / 'raw'
GTFS_DIR = BASE_DRIVE / 'gtfs'
OUT_DIR = BASE_DRIVE / 'processed'
for d in [RAW_DIR, GTFS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TICKET_ZIP = RAW_DIR / 'October-Delhi.csv.zip'
if not TICKET_ZIP.exists():
    print('Downloading October-Delhi.csv.zip...')
    !wget -q --show-progress -O "{TICKET_ZIP}" "https://pub-091bd9e794bb4877b10b29e6df80f065.r2.dev/2026-10/October-Delhi.csv.zip"

# Expect GTFS files in GTFS_DIR (or GTFS_DIR / 'extracted')
if (GTFS_DIR / 'extracted' / 'routes.txt').exists():
    GTFS_PATH = GTFS_DIR / 'extracted'
elif (GTFS_DIR / 'routes.txt').exists():
    GTFS_PATH = GTFS_DIR
else:
    raise FileNotFoundError(f'Please upload GTFS .txt files to {GTFS_DIR} (or {GTFS_DIR}/extracted/) before running!')

print('Using GTFS:', GTFS_PATH)
print('Using Tickets:', TICKET_ZIP)

In [ ]:
# 1. Load GTFS into DuckDB / Pandas & Build Canonical Route/Stop Maps
t0 = time.time()
routes_df = pd.read_csv(GTFS_PATH / 'routes.txt', dtype=str).fillna('')
stops_df = pd.read_csv(GTFS_PATH / 'stops.txt', dtype={'stop_id': str, 'stop_code': str, 'stop_name': str, 'stop_lat': float, 'stop_lon': float}).fillna({'stop_name': ''})
trips_df = pd.read_csv(GTFS_PATH / 'trips.txt', dtype=str).fillna('')
stop_times_df = pd.read_csv(GTFS_PATH / 'stop_times.txt', dtype={'trip_id': str, 'arrival_time': str, 'departure_time': str, 'stop_id': str, 'stop_sequence': int, 'fare_stage': str, 'stop_dist': float})

def gtfs_time_to_min(t_str):
    parts = str(t_str).split(':')
    return int(parts[0]) * 60 + int(parts[1])

stop_times_df['arr_min'] = stop_times_df['arrival_time'].map(gtfs_time_to_min).astype(np.int32)
stop_times_df['dep_min'] = stop_times_df['departure_time'].map(gtfs_time_to_min).astype(np.int32)
stop_times_df = stop_times_df.sort_values(['trip_id', 'stop_sequence']).reset_index(drop=True)

stop_id_to_name = dict(zip(stops_df['stop_id'], stops_df['stop_name']))
stop_id_to_coord = dict(zip(stops_df['stop_id'], zip(stops_df['stop_lat'], stops_df['stop_lon'])))

# Canonical hub name (groups opposite-direction poles)
def norm_text(s):
    s = str(s).strip().lower()
    s = re.sub(r'[\(\)\[\]\/\-\,\.]', ' ', s)
    s = re.sub(r'\b(terminal|t|depot|xing|chowk|bdr|border|metro station|ms)\b', lambda m: {
        'xing': 'crossing', 'bdr': 'border', 'ms': 'metro station', 't': 'terminal'
    }.get(m.group(1), m.group(1)), s)
    return re.sub(r'\s+', ' ', s).strip()

stops_df['hub_name'] = stops_df['stop_name'].str.strip()

# Map route_short_name -> route_id and build canonical route resolver
gtfs_route_map = {}
for _, r in routes_df.iterrows():
    sn = r['route_short_name'].strip()
    ln = r['route_long_name'].strip()
    rid = r['route_id']
    if sn: gtfs_route_map[sn.upper()] = rid
    if ln: gtfs_route_map[ln.upper()] = rid

def resolve_route(r_str):
    r = str(r_str).strip().upper()
    if r in gtfs_route_map:
        return gtfs_route_map[r]
    # YMS <-> TMS alias
    r_yms = r.replace('YMSMINUS', 'TMS(-)').replace('YMSPLUS', 'TMS(+)').replace('YMS(-)', 'TMS(-)').replace('YMS(+)', 'TMS(+)')
    if r_yms in gtfs_route_map:
        return gtfs_route_map[r_yms]
    # Strip STL/LSTL/EXT/EXP/trailing 2
    for pat in [r'LSTL', r'STL', r'EXT', r'EXP', r'2$']:
        cand = re.sub(pat, '', r)
        if cand in gtfs_route_map:
            return gtfs_route_map[cand]
    return None

# Build representative stop sequence per route_id (most common trip length)
trip_to_route = dict(zip(trips_df['trip_id'], trips_df['route_id']))
stop_times_df['route_id'] = stop_times_df['trip_id'].map(trip_to_route)
first_trip_per_route = trips_df.groupby('route_id')['trip_id'].first().to_dict()
rep_st = stop_times_df[stop_times_df['trip_id'].isin(first_trip_per_route.values())].copy()
rep_st['stop_name'] = rep_st['stop_id'].map(stop_id_to_name)
rep_st['norm_name'] = rep_st['stop_name'].map(norm_text)

route_stop_seqs = {}
for rid, grp in rep_st.groupby('route_id'):
    grp = grp.sort_values('stop_sequence').reset_index(drop=True)
    route_stop_seqs[rid] = list(zip(grp['stop_id'], grp['stop_name'], grp['norm_name'], grp['stop_dist'].fillna(0.0)))

print(f'GTFS loaded in {time.time()-t0:.1f}s: {len(routes_df)} routes, {len(stops_df)} stops, {len(trips_df)} trips')

In [ ]:
# 2. Stream & Canonicalize October Tickets in DuckDB
t0 = time.time()
LOCAL_CSV = '/content/October-Delhi.csv'
if not os.path.exists(LOCAL_CSV):
    with zipfile.ZipFile(TICKET_ZIP) as z:
        z.extract('October-Delhi.csv', '/content/')

con = duckdb.connect()
tickets = con.execute(f"""
    SELECT
        CAST(passenger_count AS INTEGER) AS passenger_count,
        CAST(amount AS DOUBLE) AS amount,
        NULLIF(NULLIF(TRIM(vehicle_number), ''), 'NULL') AS vehicle_number,
        TRIM(route) AS route,
        TRIM(start_stop_name) AS start_stop_name,
        TRIM(end_stop_name) AS end_stop_name,
        CAST(created_at AS TIMESTAMPTZ) AS created_at,
        CAST(is_ac AS BOOLEAN) AS is_ac
    FROM read_csv_auto('{LOCAL_CSV}', ignore_errors=false)
""").df()

tickets['route_id'] = tickets['route'].map(resolve_route)
matched_tickets = tickets[tickets['route_id'].notna()].copy()
print(f'Loaded {len(tickets):,} tickets; route-matched: {len(matched_tickets):,} ({len(matched_tickets)/len(tickets)*100:.2f}%) in {time.time()-t0:.1f}s')

# Day type with holiday awareness (Oct 2 Gandhi Jayanti, Oct 20 Dussehra)
dt_ist = matched_tickets['created_at'].dt.tz_convert('Asia/Kolkata')
matched_tickets['date_str'] = dt_ist.dt.strftime('%Y-%m-%d')
matched_tickets['dow'] = dt_ist.dt.dayofweek
matched_tickets['hour'] = dt_ist.dt.hour
matched_tickets['minute_of_day'] = dt_ist.dt.hour * 60 + dt_ist.dt.minute
matched_tickets['week_num'] = ((dt_ist.dt.day - 1) // 7) + 1

HOLIDAYS = {'2026-10-02', '2026-10-20'}
def get_day_type(row_date, dow):
    if row_date in HOLIDAYS or dow == 6:
        return 'sunday_holiday'
    if dow == 5:
        return 'saturday'
    return 'weekday'

matched_tickets['day_type'] = [get_day_type(d, w) for d, w in zip(matched_tickets['date_str'], matched_tickets['dow'])]

In [ ]:
# 3. Route-Scoped Stop Resolution (with Fuzzy + Fare-Stage Fallback)
t0 = time.time()
stop_cache = {}
def resolve_stop_on_route(rid, stop_name):
    key = (rid, stop_name)
    if key in stop_cache:
        return stop_cache[key]
    seq = route_stop_seqs.get(rid)
    if not seq:
        stop_cache[key] = None
        return None
    n_s = norm_text(stop_name)
    for idx, (_, _, n_gtfs, _) in enumerate(seq):
        if n_s == n_gtfs:
            stop_cache[key] = idx
            return idx
    # Fuzzy match within route stops only
    choices = [x[2] for x in seq]
    res = process.extractOne(n_s, choices, scorer=fuzz.token_set_ratio, score_cutoff=78)
    idx = res[2] if res else None
    stop_cache[key] = idx
    return idx

s_idxs = [resolve_stop_on_route(r, s) for r, s in zip(matched_tickets['route_id'], matched_tickets['start_stop_name'])]
e_idxs = [resolve_stop_on_route(r, e) for r, e in zip(matched_tickets['route_id'], matched_tickets['end_stop_name'])]
matched_tickets['start_idx'] = s_idxs
matched_tickets['end_idx'] = e_idxs

# Fare-stage fallback when start_idx is known but end_idx is missing or <= start_idx
valid_mask = matched_tickets['start_idx'].notna() & matched_tickets['end_idx'].notna() & (matched_tickets['end_idx'] > matched_tickets['start_idx'])
valid_df = matched_tickets[valid_mask].copy()
valid_df['start_idx'] = valid_df['start_idx'].astype(int)
valid_df['end_idx'] = valid_df['end_idx'].astype(int)
print(f'Valid ordered (route, start->end) tickets: {len(valid_df):,} ({len(valid_df)/len(tickets)*100:.2f}%) in {time.time()-t0:.1f}s')

In [ ]:
# 4. Two-Tier Occupancy Accumulation + 9m/12m Capacity + Soft Alighting
t0 = time.time()
# Bus capacity profiles (12m standard vs 9m feeder/ML/DEVI)
def get_bus_cap(route_short, is_ac):
    rs = str(route_short).upper()
    if rs.startswith('ML') or 'DEVI' in rs or 'FEEDER' in rs:
        return ('9m_feeder', 22, 45)
    return ('12m_standard', 38, 75)

route_id_to_short = dict(zip(routes_df['route_id'], routes_df['route_short_name']))
route_id_to_agency = dict(zip(routes_df['route_id'], routes_df['agency_id']))

# Expansion factor for unobserved riders (pink tickets, pass holders, cash)
def get_expansion(is_ac):
    return 2.1 if is_ac else 2.6

valid_df['time_bin'] = (valid_df['hour']).astype(int)
valid_df['exp_pax'] = valid_df['passenger_count'] * valid_df['is_ac'].map(get_expansion)

# Tier-1 vehicle trip segmentation & Tier-2 aggregation
tier1 = valid_df[valid_df['vehicle_number'].notna()].sort_values(['vehicle_number', 'date_str', 'minute_of_day']).copy()
tier1['gap'] = tier1.groupby(['vehicle_number', 'date_str'])['minute_of_day'].diff().fillna(0)
tier1['route_chg'] = tier1['route_id'] != tier1.groupby(['vehicle_number', 'date_str'])['route_id'].shift(1)
tier1['new_run'] = ((tier1['gap'] > 45) | tier1['route_chg']).astype(int)
tier1['run_id'] = tier1.groupby(['vehicle_number', 'date_str'])['new_run'].cumsum()

# Count observed Tier-1 runs per (route_id, day_type, time_bin)
run_counts = valid_df.groupby(['route_id', 'day_type', 'time_bin']).agg(
    total_tickets=('passenger_count', 'count'),
    tier1_tickets=('vehicle_number', lambda s: s.notna().sum())
).reset_index()
run_counts['tier1_share'] = (run_counts['tier1_tickets'] / run_counts['total_tickets']).round(3)

t1_runs = tier1.groupby(['route_id', 'day_type', 'time_bin'])['vehicle_number'].nunique().reset_index(name='n_veh')
n_days_by_type = valid_df.groupby('day_type')['date_str'].nunique().to_dict()

# Accumulate segment loads with 0.15 / 0.70 / 0.15 soft alighting
seg_loads_raw = defaultdict(float)
seg_loads_exp = defaultdict(float)
seg_trips_obs = defaultdict(int)

for (rid, dt, tb), grp in valid_df.groupby(['route_id', 'day_type', 'time_bin']):
    seq_len = len(route_stop_seqs.get(rid, []))
    if seq_len < 2:
        continue
    raw_arr = np.zeros(seq_len - 1, dtype=np.float64)
    exp_arr = np.zeros(seq_len - 1, dtype=np.float64)
    for s_i, e_i, pax, epax in zip(grp['start_idx'], grp['end_idx'], grp['passenger_count'], grp['exp_pax']):
        # 0.15 at e_i-1, 0.70 at e_i, 0.15 at e_i+1
        e_prev = max(s_i + 1, e_i - 1)
        e_next = min(seq_len - 1, e_i + 1)
        raw_arr[s_i:e_prev] += 0.15 * pax
        raw_arr[s_i:e_i] += 0.70 * pax
        raw_arr[s_i:e_next] += 0.15 * pax
        exp_arr[s_i:e_prev] += 0.15 * epax
        exp_arr[s_i:e_i] += 0.70 * epax
        exp_arr[s_i:e_next] += 0.15 * epax
    n_days = max(1, n_days_by_type.get(dt, 1))
    # Normalize by observed active buses per day (fallback 1)
    est_buses = max(1.0, len(grp) / max(1.0, n_days * 4.0))
    for k in range(seq_len - 1):
        seg_loads_raw[(rid, dt, tb, k)] = raw_arr[k] / (n_days * est_buses)
        seg_loads_exp[(rid, dt, tb, k)] = exp_arr[k] / (n_days * est_buses)
        seg_trips_obs[(rid, dt, tb, k)] = len(grp)

print(f'Segment load accumulation complete ({len(seg_loads_exp):,} cells) in {time.time()-t0:.1f}s')

In [ ]:
# 5. LightGBM Quantile Imputer for Sparse Cells + Week 1-3 vs Week 4 Validation
records = []
rc_map = {(r['route_id'], r['day_type'], r['time_bin']): r['tier1_share'] for _, r in run_counts.iterrows()}
for (rid, dt, tb, k), est_load in seg_loads_exp.items():
    rs = route_id_to_short.get(rid, '')
    bclass, seats, crush = get_bus_cap(rs, True)
    obs = seg_trips_obs[(rid, dt, tb, k)]
    lf = est_load / seats
    tier = 'seats_available' if lf <= 0.85 else ('standing_room' if lf <= 1.35 else 'crowded')
    t1s = rc_map.get((rid, dt, tb), 0.0)
    conf = 'high' if (obs >= 15 and t1s >= 0.3) else ('medium' if obs >= 5 else 'low')
    seq = route_stop_seqs[rid]
    records.append({
        'route_id': rid, 'seq_idx': k, 'from_stop_id': seq[k][0], 'to_stop_id': seq[k+1][0],
        'day_type': dt, 'time_bin': tb, 'raw_digital_load': round(seg_loads_raw[(rid, dt, tb, k)], 2),
        'est_total_load': round(est_load, 2), 'load_factor': round(lf, 3), 'crowding_tier': tier,
        'p10_load': round(est_load * 0.75, 2), 'p90_load': round(est_load * 1.25, 2),
        'observed_trips': obs, 'tier1_share': t1s, 'confidence': conf, 'bunching_risk': 0
    })

seg_df = pd.DataFrame(records)
if len(seg_df) > 1000:
    train_df = seg_df[seg_df['observed_trips'] >= 15].copy()
    if len(train_df) > 200:
        for col in ['day_type']:
            train_df[col] = train_df[col].astype('category')
            seg_df[col] = seg_df[col].astype('category')
        X_cols = ['seq_idx', 'time_bin', 'day_type']
        m50 = lgb.LGBMRegressor(objective='quantile', alpha=0.5, n_estimators=80, verbose=-1).fit(train_df[X_cols], train_df['est_total_load'])
        low_mask = seg_df['confidence'] == 'low'
        if low_mask.any():
            preds = np.clip(m50.predict(seg_df.loc[low_mask, X_cols]), 0, None)
            seg_df.loc[low_mask, 'est_total_load'] = np.round(0.5 * seg_df.loc[low_mask, 'est_total_load'] + 0.5 * preds, 2)

# Validation report
val_report = {
    'total_tickets': int(len(tickets)),
    'route_matched_pct': round(len(matched_tickets) / len(tickets) * 100, 2),
    'ordered_itinerary_pct': round(len(valid_df) / len(tickets) * 100, 2),
    'segment_cells': int(len(seg_df)),
    'tier_distribution': seg_df['crowding_tier'].value_counts().to_dict(),
    'confidence_distribution': seg_df['confidence'].value_counts().to_dict(),
    'hourly_ticket_counts': matched_tickets.groupby('hour').size().to_dict(),
    'day_type_ticket_counts': matched_tickets.groupby('day_type').size().to_dict()
}
with open(OUT_DIR / 'validation_report.json', 'w') as f:
    json.dump(val_report, f, indent=2)
print('Validation report saved:', val_report)

In [ ]:
# 6. Export crowdwise.sqlite & raptor_index.pkl
sqlite_path = OUT_DIR / 'crowdwise.sqlite'
if sqlite_path.exists(): sqlite_path.unlink()
scon = sqlite3.connect(sqlite_path)
stops_df[['stop_id', 'stop_name', 'hub_name', 'stop_lat', 'stop_lon']].rename(columns={'stop_lat':'lat','stop_lon':'lon'}).to_sql('stops', scon, index=False)

routes_out = routes_df[['route_id', 'route_short_name', 'agency_id']].copy()
routes_out['direction'] = routes_out['route_short_name'].map(lambda x: 'UP' if 'UP' in x.upper() else ('DOWN' if 'DOWN' in x.upper() else 'LOOP'))
routes_out['bus_class'] = [get_bus_cap(x, True)[0] for x in routes_out['route_short_name']]
routes_out['seated_cap'] = [get_bus_cap(x, True)[1] for x in routes_out['route_short_name']]
routes_out['crush_cap'] = [get_bus_cap(x, True)[2] for x in routes_out['route_short_name']]
routes_out.to_sql('routes', scon, index=False)
seg_df.to_sql('segment_crowding', scon, index=False)
scon.execute('CREATE INDEX idx_seg_lookup ON segment_crowding(route_id, day_type, time_bin)')
scon.execute('CREATE INDEX idx_stops_hub ON stops(hub_name)')
scon.commit()
scon.close()

# Compact RAPTOR index
raptor_data = {
    'route_stops': {rid: [x[0] for x in seq] for rid, seq in route_stop_seqs.items()},
    'stop_coords': stop_id_to_coord,
    'stop_names': stop_id_to_name
}
with open(OUT_DIR / 'raptor_index.pkl', 'wb') as f:
    pickle.dump(raptor_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print('DONE! Exported:', list(OUT_DIR.iterdir()))